# BA and pooled-region stress event catalog

Work through four stress-event calculations on a small toy example, then apply them to four historical regions: SPP (`SWPP`), MISO subregion 8910 (`MISO_8910`), MISO subregion sum (`MISO_SUBREGION_SUM`), and the WECC pool. Geographic memberships can overlap. The toy example defines its event table directly; the historical workflow first converts scenario metrics to that table, then reuses the same four calculations.

**Inputs:** released 2007–2023 scenario metrics and their BA/pooled-region manifests. The toy inputs are defined directly below; the historical functions also apply to other manifest-listed regions.

**Requirements:** The repository environment and bundled inputs. Both the toy example and the four regional examples run without downloads or a weather service.

**Outputs:** four complete regional event CSVs under `data_outputs/data_flow/ba_stress_event_catalog/`, with inline thresholds, risk hours, toy events, and historical summaries. These example CSVs do not overwrite the released catalogs.

Run the code cells from top to bottom. See the [setup instructions](../../README.md#quick-start).

Load thresholds use the load series; net-load thresholds share the 90th, 95th, and 99th percentiles across portfolios within each region. Renewable CF uses absolute 10%, 5%, and 1% thresholds. Qualifying hours separated by one non-risk hour form one bridged event, with the skipped hour recorded in `gap_hours`.


## Setup

Load the libraries, select the output directory, and retain the released event-column schema.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

OUTPUT_DIR = Path("../../data_outputs/data_flow/ba_stress_event_catalog")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

EVENT_COLUMNS = [
    "ba_code",
    "metric",
    "threshold_quantile",
    "event_start_utc",
    "event_end_utc",
    "event_length_hours",
    "gap_hours",
    "event_peak_value",
    "event_peak_quantile",
    "scenario_specific_event_peak_quantile",
    "event_peak_utc",
]


## Step 1: Prepare the toy inputs and settings

Define six hours of load, two net-load portfolios, and their renewable CF values. The illustrative wind/solar labels identify the toy portfolios.


In [2]:
# Start from an event table constructed directly for the toy example.
# The historical workflow below creates the same table from scenario metrics.
# The wind/solar split labels are illustrative scenario names for the tiny example.
example_event_table = pd.DataFrame(
    [
        ("2023-01-01T00:00:00Z", "load_mw", 80),
        ("2023-01-01T00:00:00Z", "net_load_mw__wind25_solar75", 26),
        ("2023-01-01T00:00:00Z", "net_load_mw__wind50_solar50", 24),
        ("2023-01-01T00:00:00Z", "renew_cf_equiv__wind25_solar75", 0.54),
        ("2023-01-01T00:00:00Z", "renew_cf_equiv__wind50_solar50", 0.56),
        ("2023-01-01T01:00:00Z", "load_mw", 95),
        ("2023-01-01T01:00:00Z", "net_load_mw__wind25_solar75", 65),
        ("2023-01-01T01:00:00Z", "net_load_mw__wind50_solar50", 63),
        ("2023-01-01T01:00:00Z", "renew_cf_equiv__wind25_solar75", 0.30),
        ("2023-01-01T01:00:00Z", "renew_cf_equiv__wind50_solar50", 0.32),
        ("2023-01-01T02:00:00Z", "load_mw", 96),
        ("2023-01-01T02:00:00Z", "net_load_mw__wind25_solar75", 95),
        ("2023-01-01T02:00:00Z", "net_load_mw__wind50_solar50", 94),
        ("2023-01-01T02:00:00Z", "renew_cf_equiv__wind25_solar75", 0.01),
        ("2023-01-01T02:00:00Z", "renew_cf_equiv__wind50_solar50", 0.02),
        ("2023-01-01T03:00:00Z", "load_mw", 84),
        ("2023-01-01T03:00:00Z", "net_load_mw__wind25_solar75", 75),
        ("2023-01-01T03:00:00Z", "net_load_mw__wind50_solar50", 70),
        ("2023-01-01T03:00:00Z", "renew_cf_equiv__wind25_solar75", 0.09),
        ("2023-01-01T03:00:00Z", "renew_cf_equiv__wind50_solar50", 0.14),
        ("2023-01-01T04:00:00Z", "load_mw", 97),
        ("2023-01-01T04:00:00Z", "net_load_mw__wind25_solar75", 95),
        ("2023-01-01T04:00:00Z", "net_load_mw__wind50_solar50", 93),
        ("2023-01-01T04:00:00Z", "renew_cf_equiv__wind25_solar75", 0.02),
        ("2023-01-01T04:00:00Z", "renew_cf_equiv__wind50_solar50", 0.04),
        ("2023-01-01T05:00:00Z", "load_mw", 98),
        ("2023-01-01T05:00:00Z", "net_load_mw__wind25_solar75", 97),
        ("2023-01-01T05:00:00Z", "net_load_mw__wind50_solar50", 96),
        ("2023-01-01T05:00:00Z", "renew_cf_equiv__wind25_solar75", 0.01),
        ("2023-01-01T05:00:00Z", "renew_cf_equiv__wind50_solar50", 0.02),
    ],
    columns=["time_utc", "metric", "value"],
)
example_event_table["time_utc"] = pd.to_datetime(example_event_table["time_utc"], utc=True)


example_risk_metric_settings = pd.DataFrame(
    [
        ("load_mw", "load_mw", [0.75, 0.90], "max"),
        ("net_load_mw__wind25_solar75", "net_load_mw", [0.75, 0.90], "max"),
        ("net_load_mw__wind50_solar50", "net_load_mw", [0.75, 0.90], "max"),
        ("renew_cf_equiv__wind25_solar75", "renew_cf_equiv", [0.25, 0.10], "min"),
        ("renew_cf_equiv__wind50_solar50", "renew_cf_equiv", [0.25, 0.10], "min"),
    ],
    columns=["metric", "risk_metric_type", "threshold_quantiles", "tie_rank_method"],
)

## Step 2: Work through the shared event calculations

Event table → quantile ranks → thresholds → risk hours → stress events. Each function is followed by its toy calculation and result.


### Calculate quantile ranks

In [3]:
def calculate_quantile_ranks(hourly_metrics, risk_metric_settings):
    """Add risk-type settings and quantile-rank context for one region."""

    # Attach the visible settings table to each hourly row using the exact metric name.
    hourly_metrics = hourly_metrics.merge(risk_metric_settings, on="metric", how="left")

    # Require explicit settings for every metric included in the event table.
    unmatched_metrics = hourly_metrics.loc[hourly_metrics["risk_metric_type"].isna(), "metric"].unique()
    if len(unmatched_metrics) > 0:
        raise ValueError(f"Some metrics did not match risk_metric_settings: {sorted(unmatched_metrics)}")

    # Rank within each risk type, then within each individual metric series.
    for risk_metric_type, risk_type_values in hourly_metrics.groupby("risk_metric_type"):
        tie_rank_method = risk_type_values["tie_rank_method"].iloc[0]
        hourly_metrics.loc[risk_type_values.index, "event_peak_quantile"] = risk_type_values["value"].rank(method=tie_rank_method, pct=True)

    # Scenario-specific peak ranks retain the original metric series as their population.
    for metric, metric_values in hourly_metrics.groupby("metric"):
        tie_rank_method = metric_values["tie_rank_method"].iloc[0]
        hourly_metrics.loc[metric_values.index, "scenario_specific_event_peak_quantile"] = (metric_values["value"].rank(method=tie_rank_method, pct=True))

    return hourly_metrics

In [4]:
# Calculate quantile ranks for the toy event table.
example_ranked = calculate_quantile_ranks(example_event_table, example_risk_metric_settings)


#### Display: toy quantile ranks

Ranks use each risk type as one population, then each individual metric series as its own population.


In [5]:
# Display: toy values and their two quantile ranks.
preview = example_ranked[["metric", "time_utc", "value", "event_peak_quantile", "scenario_specific_event_peak_quantile"]]
preview = preview.rename(
    columns={
        "metric": "Metric",
        "time_utc": "Time (UTC)",
        "value": "Value (MW or CF fraction)",
        "event_peak_quantile": "Rank within risk type (fraction)",
        "scenario_specific_event_peak_quantile": "Rank within scenario (fraction)",
    }
)
preview = preview.set_index(["Metric", "Time (UTC)"])
display(preview.style.format(precision=3))


,,Value (MW or CF fraction),Rank within risk type (fraction),Rank within scenario (fraction)
Metric,Time (UTC),,,
load_mw,2023-01-01 00:00:00+00:00,80.000,0.167,0.167
net_load_mw__wind25_solar75,2023-01-01 00:00:00+00:00,26.000,0.167,0.167
net_load_mw__wind50_solar50,2023-01-01 00:00:00+00:00,24.000,0.083,0.167
renew_cf_equiv__wind25_solar75,2023-01-01 00:00:00+00:00,0.540,0.917,1.000
renew_cf_equiv__wind50_solar50,2023-01-01 00:00:00+00:00,0.560,1.000,1.000
load_mw,2023-01-01 01:00:00+00:00,95.000,0.500,0.500
net_load_mw__wind25_solar75,2023-01-01 01:00:00+00:00,65.000,0.333,0.333
net_load_mw__wind50_solar50,2023-01-01 01:00:00+00:00,63.000,0.250,0.333
renew_cf_equiv__wind25_solar75,2023-01-01 01:00:00+00:00,0.300,0.750,0.833


### Calculate thresholds

For load and net load, the threshold setting is a quantile. Net-load quantiles are shared across portfolios within the region; load has one series. For renewable CF, the setting is an absolute CF cutoff. Threshold values retain the precision used in the comparisons.

In [6]:
def calculate_thresholds(hourly_metrics):
    """Calculate quantile load thresholds and absolute renewable-CF thresholds."""
    threshold_table_rows = []

    # Calculate one shared threshold table for each risk metric type.
    for risk_metric_type, risk_type_values in hourly_metrics.groupby("risk_metric_type"):
        threshold_quantiles = risk_type_values["threshold_quantiles"].iloc[0]
        for threshold_quantile in threshold_quantiles:
            if risk_metric_type == "renew_cf_equiv":
                threshold_value = threshold_quantile
            else:
                threshold_value = risk_type_values["value"].quantile(threshold_quantile)

            threshold_table_rows.append(
                {
                    "risk_metric_type": risk_metric_type,
                    "threshold_quantile": threshold_quantile,
                    "threshold_value": threshold_value,
                }
            )

    return pd.DataFrame(threshold_table_rows)

In [7]:
# Calculate thresholds from the ranked toy values.
example_thresholds = calculate_thresholds(example_ranked)


#### Display: toy thresholds

Load and net-load cutoffs are in MW; renewable-CF cutoffs are dimensionless fractions.


In [8]:
# Display: toy thresholds at their comparison precision.
preview = example_thresholds.rename(
    columns={
        "risk_metric_type": "Risk measure",
        "threshold_quantile": "Threshold setting",
        "threshold_value": "Threshold value (MW or CF fraction)",
    }
)
preview = preview.set_index(["Risk measure", "Threshold setting"])
display(preview)


Threshold value (MW or CF fraction)
Risk measure   Threshold setting                                     
load_mw        0.75                                             96.75
               0.90                                             97.50
net_load_mw    0.75                                             95.00
               0.90                                             95.90
renew_cf_equiv 0.25                                              0.25
               0.10                                              0.10

### Identify risk hours

Select load/net-load values $x \geq T$ and renewable CF values $x \leq T$. The [pairwise pooling equations](../analysis/pairwise_pooling_heatmap.ipynb) also express net load relative to a threshold, using their own scenario-specific thresholds and positive-shortfall conditions.

In [9]:
def identify_risk_hours(hourly_metrics, thresholds):
    """Return hourly rows that cross each threshold for one region."""
    risk_hour_columns = [
        "metric",
        "risk_metric_type",
        "threshold_quantile",
        "threshold_value",
        "time_utc",
        "value",
        "event_peak_quantile",
        "scenario_specific_event_peak_quantile",
    ]

    # Attach each risk_metric_type threshold to every hourly row in that risk group.
    candidate_risk_hours = hourly_metrics.merge(thresholds, on="risk_metric_type")

    # Compare values with their risk-type thresholds.
    # Load/net-load are upper-tail risks; renewable-CF-equivalent is a lower-tail risk.
    is_load_netload = candidate_risk_hours["risk_metric_type"].isin(["load_mw", "net_load_mw"])
    is_renewable_cf = candidate_risk_hours["risk_metric_type"].eq("renew_cf_equiv")
    exceeds_load_threshold = candidate_risk_hours["value"].ge(candidate_risk_hours["threshold_value"])
    below_cf_threshold = candidate_risk_hours["value"].le(candidate_risk_hours["threshold_value"])
    is_load_netload_risk_hour = is_load_netload & exceeds_load_threshold
    is_renewable_cf_risk_hour = is_renewable_cf & below_cf_threshold
    is_risk_hour = is_load_netload_risk_hour | is_renewable_cf_risk_hour
    risk_hours = candidate_risk_hours.loc[is_risk_hour, risk_hour_columns].copy()
    return risk_hours.sort_values(["metric", "threshold_quantile", "time_utc"]).reset_index(drop=True)

In [10]:
# Identify toy hours crossing each threshold.
example_risk_hours = identify_risk_hours(example_ranked, example_thresholds)


#### Display: toy risk hours

A row identifies a metric, threshold setting, and qualifying hour.


In [11]:
# Display: qualifying toy hours.
preview = example_risk_hours
preview = preview.assign(value=[format(value, ".3f" if metric.startswith("renew_cf") else ".1f") for metric, value in zip(example_risk_hours["metric"], example_risk_hours["value"])])
preview = preview.rename(
    columns={
        "metric": "Metric",
        "risk_metric_type": "Risk measure",
        "threshold_quantile": "Threshold setting",
        "threshold_value": "Threshold (MW or CF fraction)",
        "time_utc": "Time (UTC)",
        "value": "Value (MW or CF fraction)",
        "event_peak_quantile": "Rank within risk type (fraction)",
        "scenario_specific_event_peak_quantile": "Rank within scenario (fraction)",
    }
)
preview = preview.set_index(["Metric", "Threshold setting", "Time (UTC)"])
preview = preview.style.format(precision=3)
display(preview)

### Build the stress-event catalog

In [12]:
def build_stress_event_catalog(risk_hours, region_code):
    """Build final stress-event catalog rows for one region."""
    if risk_hours.empty:
        return pd.DataFrame(columns=EVENT_COLUMNS)

    one_hour = pd.Timedelta(hours=1)
    risk_groups = ["metric", "threshold_quantile"]

    # Measure the time between consecutive risk hours.
    risk_hours = risk_hours.sort_values(risk_groups + ["time_utc"]).copy()
    risk_hours["time_gap"] = risk_hours.groupby(risk_groups)["time_utc"].diff()

    # Bridge one skipped hour and assign event IDs.
    risk_hours["new_event"] = risk_hours["time_gap"].isna() | risk_hours["time_gap"].gt(2 * one_hour)
    risk_hours["event_id"] = risk_hours.groupby(risk_groups)["new_event"].cumsum()
    event_groups = risk_groups + ["event_id"]

    # Calculate event spans and record bridged gap hours.
    risk_hours["gap_hour_text"] = ((risk_hours["time_utc"] - one_hour).dt.strftime("%Y-%m-%dT%H:%M:%SZ").where(risk_hours["time_gap"].eq(2 * one_hour)))

    stress_events = risk_hours.groupby(event_groups, as_index=False).agg(event_start_utc=("time_utc", "first"), event_end_utc=("time_utc", "last"), gap_hours=("gap_hour_text", lambda hours: ";".join(hours.dropna())))

    # Add 1 because an event beginning and ending at 05:00 lasts one hour.
    stress_events["event_length_hours"] = ((stress_events["event_end_utc"] - stress_events["event_start_utc"]) / one_hour).astype(int) + 1

    # Select the most stressful hour in each event.
    # Higher load is stressful; lower renewable CF is stressful.
    peak_direction = risk_hours["risk_metric_type"].map({"load_mw": 1, "net_load_mw": 1, "renew_cf_equiv": -1})
    risk_hours["peak_score"] = risk_hours["value"] * peak_direction

    # Because risk_hours is time-sorted, idxmax keeps the earliest tied peak.
    peak_indexes = risk_hours.groupby(event_groups)["peak_score"].idxmax()
    event_peaks = risk_hours.loc[
        peak_indexes,
        event_groups
        + [
            "value",
            "event_peak_quantile",
            "scenario_specific_event_peak_quantile",
            "time_utc",
        ],
    ].rename(columns={"value": "event_peak_value", "time_utc": "event_peak_utc"})

    stress_events = stress_events.merge(event_peaks, on=event_groups)
    stress_events["ba_code"] = region_code

    return (stress_events[EVENT_COLUMNS].sort_values(["metric", "threshold_quantile", "event_start_utc"]).reset_index(drop=True))

In [13]:
# Group qualifying toy hours into stress events.
example_catalog = build_stress_event_catalog(example_risk_hours, region_code="A")


#### Display: toy event catalog

One row describes the full event span, its bridged hours, and its peak.


In [14]:
# Display: labels shared by the toy and historical catalogs.
EVENT_DISPLAY_COLUMNS = {
    "ba_code": "Region code",
    "metric": "Metric",
    "threshold_quantile": "Threshold setting",
    "event_start_utc": "Event start (UTC)",
    "event_end_utc": "Event end (UTC)",
    "event_length_hours": "Event duration (h)",
    "gap_hours": "Bridged hours (UTC)",
    "event_peak_value": "Peak value (MW or CF fraction)",
    "event_peak_quantile": "Peak rank (risk type; fraction)",
    "scenario_specific_event_peak_quantile": "Peak rank (scenario; fraction)",
    "event_peak_utc": "Peak time (UTC)",
}
# Display: toy stress events.
preview = example_catalog
preview = preview.assign(event_peak_value=[format(value, ".3f" if metric.startswith("renew_cf") else ".1f") for metric, value in zip(example_catalog["metric"], example_catalog["event_peak_value"])])
preview = preview.rename(columns=EVENT_DISPLAY_COLUMNS)
preview = preview.set_index("Metric")
preview = preview.style.format(precision=3)
display(preview)

,Region code,Threshold setting,Event start (UTC),Event end (UTC),Event duration (h),Bridged hours (UTC),Peak value (MW or CF fraction),Peak rank (risk type; fraction),Peak rank (scenario; fraction),Peak time (UTC)
Metric,,,,,,,,,,
load_mw,A,0.750,2023-01-01 04:00:00+00:00,2023-01-01 05:00:00+00:00,2,,98.0,1.000,1.000,2023-01-01 05:00:00+00:00
load_mw,A,0.900,2023-01-01 05:00:00+00:00,2023-01-01 05:00:00+00:00,1,,98.0,1.000,1.000,2023-01-01 05:00:00+00:00
net_load_mw__wind25_solar75,A,0.750,2023-01-01 02:00:00+00:00,2023-01-01 05:00:00+00:00,4,2023-01-01T03:00:00Z,97.0,1.000,1.000,2023-01-01 05:00:00+00:00
net_load_mw__wind25_solar75,A,0.900,2023-01-01 05:00:00+00:00,2023-01-01 05:00:00+00:00,1,,97.0,1.000,1.000,2023-01-01 05:00:00+00:00
net_load_mw__wind50_solar50,A,0.750,2023-01-01 05:00:00+00:00,2023-01-01 05:00:00+00:00,1,,96.0,0.917,1.000,2023-01-01 05:00:00+00:00
net_load_mw__wind50_solar50,A,0.900,2023-01-01 05:00:00+00:00,2023-01-01 05:00:00+00:00,1,,96.0,0.917,1.000,2023-01-01 05:00:00+00:00
renew_cf_equiv__wind25_solar75,A,0.100,2023-01-01 02:00:00+00:00,2023-01-01 05:00:00+00:00,4,,0.010,0.083,0.167,2023-01-01 02:00:00+00:00
renew_cf_equiv__wind25_solar75,A,0.250,2023-01-01 02:00:00+00:00,2023-01-01 05:00:00+00:00,4,,0.010,0.083,0.167,2023-01-01 02:00:00+00:00
renew_cf_equiv__wind50_solar50,A,0.100,2023-01-01 02:00:00+00:00,2023-01-01 05:00:00+00:00,4,2023-01-01T03:00:00Z,0.020,0.250,0.167,2023-01-01 02:00:00+00:00


The toy catalog distinguishes individual qualifying hours from event spans. A four-hour event can include a bridged non-risk hour; `gap_hours` records that hour. High load/net load selects the maximum, while low renewable CF selects the minimum; tied peaks retain the earliest hour.

## Step 3: Read the historical inputs

The worked run uses `SWPP`, `MISO_8910`, `MISO_SUBREGION_SUM`, and `WECC` as examples. The historical loop first converts scenario metrics into the event table defined directly in the toy example. It then reuses the same four calculations: ranks, thresholds, risk hours, and events. The `REGION_INPUTS` mapping selects bundled version `0.1.0` inputs under `data_inputs/examples/`. Each keeps all historical hours and scenarios, with the five columns used below. The input table below shows those bundled paths; `metadata.source_path` retains each original archive path for provenance. To add another manifest-listed region, obtain its file from the full archive and add its code and path to that mapping.

[Pooled scenario metrics generation](pooled_scenario_metrics_generation.ipynb) explains how pooled inputs are constructed using the five-member `MISO_NCA` as its worked example. The `MISO_SUBREGION_SUM` input used here includes all six MISO subregions.


In [15]:
# These four complete paths are the inputs for the historical worked examples.
REGION_INPUTS = {
    "SWPP": Path("../../data_inputs/examples/wtk_bchrrr_nsrdb_2007_2023/ba_scenario_metrics/SWPP_wtk_bchrrr_nsrdb_2007_2023_scenario_metrics.csv.gz"),
    "MISO_8910": Path("../../data_inputs/examples/wtk_bchrrr_nsrdb_2007_2023/ba_scenario_metrics/MISO_8910_wtk_bchrrr_nsrdb_2007_2023_scenario_metrics.csv.gz"),
    "MISO_SUBREGION_SUM": Path("../../data_inputs/examples/wtk_bchrrr_nsrdb_2007_2023/pooled_scenario_metrics/MISO_SUBREGION_SUM_wtk_bchrrr_nsrdb_2007_2023_scenario_metrics.csv.gz"),
    "WECC": Path("../../data_inputs/examples/wtk_bchrrr_nsrdb_2007_2023/pooled_scenario_metrics/WECC_wtk_bchrrr_nsrdb_2007_2023_scenario_metrics.csv.gz"),
}
ba_manifest = pd.read_csv("../../manifests/ba_scenario_metric_metadata_manifest_wtk_bchrrr_nsrdb_2007_2023.csv", usecols=["ba_code", "metric", "scenario", "source_path"])
pooled_manifest = pd.read_csv("../../manifests/pooled_region_scenario_metric_metadata_manifest_wtk_bchrrr_nsrdb_2007_2023.csv", usecols=["ba_code", "metric", "scenario", "source_path"])
metadata = pd.concat([ba_manifest, pooled_manifest], ignore_index=True)

RISK_METRIC_SETTINGS = pd.DataFrame(
    [
        ("load_mw", "load_mw", [0.90, 0.95, 0.99], "max"),
        ("net_load_mw__installed_2024", "net_load_mw", [0.90, 0.95, 0.99], "max"),
        ("net_load_mw__split_w00_s100", "net_load_mw", [0.90, 0.95, 0.99], "max"),
        ("net_load_mw__split_w25_s75", "net_load_mw", [0.90, 0.95, 0.99], "max"),
        ("net_load_mw__split_w50_s50", "net_load_mw", [0.90, 0.95, 0.99], "max"),
        ("net_load_mw__split_w75_s25", "net_load_mw", [0.90, 0.95, 0.99], "max"),
        ("net_load_mw__split_w100_s00", "net_load_mw", [0.90, 0.95, 0.99], "max"),
        ("renew_cf_equiv__installed_2024", "renew_cf_equiv", [0.10, 0.05, 0.01], "min"),
        ("renew_cf_equiv__split_w00_s100", "renew_cf_equiv", [0.10, 0.05, 0.01], "min"),
        ("renew_cf_equiv__split_w25_s75", "renew_cf_equiv", [0.10, 0.05, 0.01], "min"),
        ("renew_cf_equiv__split_w50_s50", "renew_cf_equiv", [0.10, 0.05, 0.01], "min"),
        ("renew_cf_equiv__split_w75_s25", "renew_cf_equiv", [0.10, 0.05, 0.01], "min"),
        ("renew_cf_equiv__split_w100_s00", "renew_cf_equiv", [0.10, 0.05, 0.01], "min"),
    ],
    columns=["metric", "risk_metric_type", "threshold_quantiles", "tie_rank_method"],
)

metadata = metadata.merge(RISK_METRIC_SETTINGS[["metric", "risk_metric_type"]], on="metric", how="left")
metadata = metadata[metadata["ba_code"].isin(REGION_INPUTS)].copy()
missing = sorted(set(REGION_INPUTS) - set(metadata["ba_code"]))
if missing:
    raise AssertionError(f"Missing manifest rows for {missing}.")


### Display: historical input files

These are the bundled files used below; the manifests retain their original archive paths.


In [16]:
# Display: bundled files actually read by the historical loop.
preview = metadata.groupby("ba_code").agg(metrics=("metric", "size"))
for region_code, source_path in REGION_INPUTS.items():
    preview.loc[region_code, "source"] = source_path.relative_to(Path("../..")).as_posix()
preview = preview.rename_axis("Region code")
preview = preview.rename(columns={"metrics": "Metrics", "source": "Input path (repository-relative)"})
with pd.option_context("display.max_colwidth", None):
    display(preview)


,Metrics,Input path (repository-relative)
Region code,,
MISO_8910,13,data_inputs/examples/wtk_bchrrr_nsrdb_2007_2023/ba_scenario_metrics/MISO_8910_wtk_bchrrr_nsrdb_2007_2023_scenario_metrics.csv.gz
MISO_SUBREGION_SUM,13,data_inputs/examples/wtk_bchrrr_nsrdb_2007_2023/pooled_scenario_metrics/MISO_SUBREGION_SUM_wtk_bchrrr_nsrdb_2007_2023_scenario_metrics.csv.gz
SWPP,13,data_inputs/examples/wtk_bchrrr_nsrdb_2007_2023/ba_scenario_metrics/SWPP_wtk_bchrrr_nsrdb_2007_2023_scenario_metrics.csv.gz
WECC,13,data_inputs/examples/wtk_bchrrr_nsrdb_2007_2023/pooled_scenario_metrics/WECC_wtk_bchrrr_nsrdb_2007_2023_scenario_metrics.csv.gz


### Convert scenario metrics to the event table

In [17]:
def scenario_metrics_to_event_table(scenario_metrics, scenario_metric_metadata):
    """Convert scenario-metrics rows into the event-pipeline table for isolated BAs and BA pools."""
    scenario_metrics = scenario_metrics.assign(time_utc=pd.to_datetime(scenario_metrics["time_utc"], utc=True))
    event_table_frames = []

    for _, metric_spec in scenario_metric_metadata.iterrows():
        catalog_metric = metric_spec["metric"]
        scenario = metric_spec["scenario"]
        value_column = metric_spec["risk_metric_type"]

        scenario_rows = scenario_metrics.loc[scenario_metrics["scenario"].eq(scenario), ["time_utc", value_column]]
        if scenario_rows.empty:
            raise AssertionError(f"Missing scenario rows for {catalog_metric}: {scenario}")

        event_rows = scenario_rows.rename(columns={value_column: "value"})
        event_rows["metric"] = catalog_metric
        event_table_frames.append(event_rows[["metric", "time_utc", "value"]])

    if not event_table_frames:
        raise AssertionError("No event-table rows were created.")

    return pd.concat(event_table_frames, ignore_index=True)

## Step 4: Build the historical event catalogs

In [18]:
catalog_preview_frames = []
summary_rows = []

for region_code, source_path in REGION_INPUTS.items():
    print(f"Processing {region_code}: {source_path.as_posix()}")
    region_metadata = metadata[metadata["ba_code"].eq(region_code)].copy()
    scenario_metrics = pd.read_csv(source_path, usecols=["time_utc", "scenario", "load_mw", "net_load_mw", "renew_cf_equiv"])

    # Prepare the event table from historical scenario metrics.
    region_event_table = scenario_metrics_to_event_table(scenario_metrics, region_metadata)

    # Shared calculation 1: Calculate quantile ranks.
    region_ranked = calculate_quantile_ranks(region_event_table, RISK_METRIC_SETTINGS)

    # Shared calculation 2: Calculate thresholds.
    region_thresholds = calculate_thresholds(region_ranked)

    # Shared calculation 3: Identify risk hours.
    region_risk_hours = identify_risk_hours(region_ranked, region_thresholds)

    # Shared calculation 4: Build the stress-event catalog.
    region_catalog = build_stress_event_catalog(region_risk_hours, region_code=region_code)
    if region_catalog.empty:
        raise AssertionError(f"No events were created for {region_code}.")

    # Write the complete event catalog.
    output_path = OUTPUT_DIR / f"{region_code}_wtk_bchrrr_nsrdb_2007_2023_events.csv"
    region_catalog.to_csv(output_path, index=False, date_format="%Y-%m-%dT%H:%M:%SZ")
    print(f"Wrote {len(region_catalog):,} events for {region_code}: {output_path.as_posix()}")

    risk_hour_counts = region_risk_hours["risk_metric_type"].value_counts()
    summary_rows.append(
        {
            "region": region_code,
            "hourly_metric_rows": len(region_event_table),
            "load_risk_hours": int(risk_hour_counts.get("load_mw", 0)),
            "net_load_risk_hours": int(risk_hour_counts.get("net_load_mw", 0)),
            "renewable_cf_risk_hours": int(risk_hour_counts.get("renew_cf_equiv", 0)),
            "events": len(region_catalog),
            "bridged_events": int(region_catalog["gap_hours"].ne("").sum()),
        }
    )
    catalog_preview_frames.append(region_catalog.head(2).copy())
    del scenario_metrics, region_event_table, region_ranked, region_thresholds, region_risk_hours

catalog_preview = pd.concat(catalog_preview_frames, ignore_index=True)
catalog_summary = pd.DataFrame(summary_rows)


Processing SWPP: ../../data_inputs/examples/wtk_bchrrr_nsrdb_2007_2023/ba_scenario_metrics/SWPP_wtk_bchrrr_nsrdb_2007_2023_scenario_metrics.csv.gz


Wrote 57,160 events for SWPP: ../../data_outputs/data_flow/ba_stress_event_catalog/SWPP_wtk_bchrrr_nsrdb_2007_2023_events.csv
Processing MISO_8910: ../../data_inputs/examples/wtk_bchrrr_nsrdb_2007_2023/ba_scenario_metrics/MISO_8910_wtk_bchrrr_nsrdb_2007_2023_scenario_metrics.csv.gz


Wrote 122,585 events for MISO_8910: ../../data_outputs/data_flow/ba_stress_event_catalog/MISO_8910_wtk_bchrrr_nsrdb_2007_2023_events.csv
Processing MISO_SUBREGION_SUM: ../../data_inputs/examples/wtk_bchrrr_nsrdb_2007_2023/pooled_scenario_metrics/MISO_SUBREGION_SUM_wtk_bchrrr_nsrdb_2007_2023_scenario_metrics.csv.gz


Wrote 57,251 events for MISO_SUBREGION_SUM: ../../data_outputs/data_flow/ba_stress_event_catalog/MISO_SUBREGION_SUM_wtk_bchrrr_nsrdb_2007_2023_events.csv
Processing WECC: ../../data_inputs/examples/wtk_bchrrr_nsrdb_2007_2023/pooled_scenario_metrics/WECC_wtk_bchrrr_nsrdb_2007_2023_scenario_metrics.csv.gz


Wrote 57,731 events for WECC: ../../data_outputs/data_flow/ba_stress_event_catalog/WECC_wtk_bchrrr_nsrdb_2007_2023_events.csv


### Display: regional summary

Counts include every scenario and threshold; they are qualifying rows rather than unique clock hours.


In [19]:
# Display: qualifying rows and event counts by region.
REGION_LABELS = {
    "SWPP": "SPP",
    "MISO_8910": "MISO subregion 8910",
    "MISO_SUBREGION_SUM": "MISO subregion sum",
    "WECC": "WECC pool",
}
preview = catalog_summary
preview = preview.replace({"region": REGION_LABELS})
preview = preview.rename(
    columns={
        "region": "Region",
        "hourly_metric_rows": "Hourly metric rows",
        "load_risk_hours": "Load qualifying rows",
        "net_load_risk_hours": "Net-load qualifying rows",
        "renewable_cf_risk_hours": "Renewable-CF qualifying rows",
        "events": "Events",
        "bridged_events": "Bridged events",
    }
)
preview = preview.set_index("Region")
display(preview)


,Hourly metric rows,Load qualifying rows,Net-load qualifying rows,Renewable-CF qualifying rows,Events,Bridged events
Region,,,,,,
SPP,1935960,23828,142964,288909,57160,811
MISO subregion 8910,1935960,23828,142964,774574,122585,12746
MISO subregion sum,1935960,23828,142964,306961,57251,819
WECC pool,1935960,23828,142964,264192,57731,505


### Display: historical catalog examples

Show two events from each region; the exported CSVs contain the complete catalogs.


In [20]:
# Display: two events from each of the four regions.
preview = catalog_preview
preview = preview.assign(event_peak_value=[format(value, ".3f" if metric.startswith("renew_cf") else ".1f") for metric, value in zip(catalog_preview["metric"], catalog_preview["event_peak_value"])])
preview = preview.replace({"ba_code": REGION_LABELS})
preview = preview.rename(columns=EVENT_DISPLAY_COLUMNS)
preview = preview.rename(columns={"Region code": "Region"})
preview = preview.set_index(["Region", "Metric"])
preview = preview.style.format(precision=3)
display(preview)

## Interpretation

The summary reports qualifying rows separately for each risk type. A timestamp may qualify under several scenarios and thresholds, so these counts are not unique clock hours. The event CSVs are reconstructed examples; they do not overwrite the deposited catalogs. The four regions include individual BAs/subregions and pools, whose memberships can overlap.

## Appendix A: State-level application

The deposited state catalogs use `load_mw__raw`, `net_load_mw__raw`, and renewable-equivalent capacity factor. These raw series retain TELL/TaiESM1 hourly weather variability without GCAM annual-demand scaling; the complete raw and eight GCAM-scaled trajectories remain in the packaged state scenario metrics. [State load generation](state_load_generation.ipynb) and Appendix A of [site CF generation and weighting](site_cf_generation_ba_weighting_validation.ipynb) document the corresponding inputs.

If the GCAM trajectories are later used to select comparable stress periods, use one fixed raw-reference P95 across the cases rather than recalculating a percentile for each GCAM case. This preserves the effect of long-term load growth instead of normalizing it away. The fixed threshold is a diagnostic reference, not an adequacy standard.